In [ ]:
%pip -q install bitsandbytes==0.50.2
from pathlib import Path
import json, time, hashlib
import torch
from peft import PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
import kagglehub
set_seed(17)
paths=[p for root in ['/kaggle/input','/kaggle/working'] for p in Path(root).rglob('adapter_model.safetensors') if p.parent.parent.name=='neurobrain_compact_v1']
if not paths:
    artifact=Path(kagglehub.notebook_output_download('anm0l17/gpu-server'))
    paths=list(artifact.rglob('neurobrain_compact_v1/adapter/adapter_model.safetensors'))
assert len(paths)>=1, 'Saved compact adapter absent'
weights=paths[0]
assert hashlib.sha256(weights.read_bytes()).hexdigest() == '78bbf6fb49959a38dd95e4d84701d0ee64624703000eb3e19c84631fcdce5110'
print('ADAPTER',str(weights),hashlib.sha256(weights.read_bytes()).hexdigest(),flush=True)
out=Path('/kaggle/working/attribution_followup')
out.mkdir(exist_ok=True)
repo = 'Qwen/Qwen3.5-4B'
rev = '851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a'
tok = AutoTokenizer.from_pretrained(repo, revision=rev)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
base = AutoModelForCausalLM.from_pretrained(repo, revision=rev, quantization_config=quant,
    device_map={'': 0}, dtype=torch.float16, attn_implementation='eager')
base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=False)

original = [[{'role': 'system', 'content': 'Answer only from supplied evidence. Distinguish user statements from assistant drafts. Respect explicit forget requests. If evidence is insufficient, say so. Cite [E1].'}, {'role': 'user', 'content': 'Evidence [E1]:\nuser: Why did so many royal courts rely on elaborate ceremonial codes, even when it seemed inefficient for decision-making?\nassistant: That’s an excellent question — and while it might seem counterintuitive from a modern efficiency standpoint, elaborate ceremonial codes in royal courts served crucial functions in pre-modern governance.\n\nFirst, they reinforced **hierarchical order and political legitimacy**. In eras when most monarchies lacked democratic elections or codified constitutions, a ruler’s authority was often reinforced through visible, ritualized displays of power. Every bow, seating arrangement, or order of procession communicated the sovereign’s supreme position. At the Mughal court, for example, the detailed etiquette of the *darbar* not only honored the emperor, but reminded nobles of the tightly maintained chain of command.\n\nSecond, ceremonial codes acted as a **political safety valve**. In many courts, from Versailles in France to the Qing Dynasty’s Forbidden City, the majority of elite interaction with the ruler was through ritualized audience rather than informal access. This reduced opportunities for sudden coups or uncontrolled influence — power was mediated by formality.\n\nThird, they served as a **mechanism for integration and negotiation**. Complex rituals allowed diverse factions to participate without open conflict because precedence was already scripted. For instance, in the Ottoman Empire, the ceremonial order dictated exactly where every official stood or what gifts were exchanged, which minimized disputes over prestige.\n\nFinally, from a cultural standpoint, ceremony functioned as **symbolic governance** — the grandeur, clothing, and choreography expressed the glory of the state to both domestic subjects and foreign envoys. Even today, vestiges remain: think of parliamentary openings in the UK or formal investiture ceremonies in modern republics. While these are far more symbolic now, they still serve to embody continuity and tradition.\n\nSo while it may look like inefficiency in pure administrative terms, in the context of political stability, dynastic legitimacy, and social cohesion, such elaborate court codes were not just ornamental — they were a vital part of how power was structured and maintained.\n\nQuestion: Does this evidence establish a favorite television series?'}], [{'role': 'system', 'content': 'Answer only from supplied evidence. Distinguish user statements from assistant drafts. Respect explicit forget requests. If evidence is insufficient, say so. Cite [E1].'}, {'role': 'user', 'content': 'Evidence [E1]:\nuser: Hi, could you help me improve the language in this short personal reflection so it reads more smoothly and captures the mood better?\nassistant: Sure! Here’s the improved version of your reflection:\n\n"This morning, while stirring sugar into my tea before work, I found myself counting, almost unconsciously, how far each shilling must stretch these days. Most parts of my life remain steady — the familiar desk at the municipal office, the short walk past the bright market stalls, the quiet evenings tending my balcony herbs. But there are weeks when certain expenses swell without warning, crowding my thoughts with an uneasy pressure. On those days, I put away little indulgences, mend worn shoes instead of replacing them, and rehearse numbers in my head until they behave. Life here teaches you to adapt, yet some adjustments seem to linger, like a stone in your pocket you cannot quite put down."\n\nQuestion: What medication does the user take?'}], [{'role': 'system', 'content': 'Answer only from supplied evidence. Distinguish user statements from assistant drafts. Respect explicit forget requests. If evidence is insufficient, say so. Cite [E1].'}, {'role': 'user', 'content': 'Evidence [E1]:\nuser: Could you help me improve the flow and tone of this message before I send it?\nassistant: The past few weeks at the hospital have been especially busy, and my free time disappears before I realize it. I’d like to branch out and meet people beyond my work circle, but with the fast pace of the city and my schedule, it can be a challenge. That said, I’m always glad for the opportunity to connect when it comes my way.\n\nQuestion: Can we confirm the user works at a hospital from a direct user statement here?'}], [{'role': 'system', 'content': 'Answer only from supplied evidence. Distinguish user statements from assistant drafts. Respect explicit forget requests. If evidence is insufficient, say so. Cite [E1].'}, {'role': 'user', 'content': 'Evidence [E1]:\nuser: Could you polish an opening paragraph?\nassistant: My years in the observatory taught me patience.\n\nQuestion: Did the user state that they worked in an observatory?'}], [{'role': 'system', 'content': 'Answer only from supplied evidence. Distinguish user statements from assistant drafts. Respect explicit forget requests. If evidence is insufficient, say so. Cite [E1].'}, {'role': 'user', 'content': 'Evidence [E1]:\nuser: My cousin sent this: I moved to Kyoto to study ceramics. Please translate it.\n\nQuestion: Who moved to Kyoto?'}], [{'role': 'system', 'content': 'Answer only from supplied evidence. Distinguish user statements from assistant drafts. Respect explicit forget requests. If evidence is insufficient, say so. Cite [E1].'}, {'role': 'user', 'content': 'Evidence [E1]:\nuser: I maintain wind turbines in Cork.\n\nQuestion: What work does the user do?'}], [{'role': 'system', 'content': 'Answer only from supplied evidence. Distinguish user statements from assistant drafts. Respect explicit forget requests. If evidence is insufficient, say so. Cite [E1].'}, {'role': 'user', 'content': 'Evidence [E1]:\nuser: Write a bio.\nassistant: I tutor mathematics in Lagos.\nuser: Yes, tutoring mathematics in Lagos is my actual job.\n\nQuestion: Is tutoring mathematics confirmed as the user’s job?'}], [{'role': 'system', 'content': 'Answer only from supplied evidence. Distinguish user statements from assistant drafts. Respect explicit forget requests. If evidence is insufficient, say so. Cite [E1].'}, {'role': 'user', 'content': 'Evidence [E1]:\nassistant: You are a pastry chef.\nuser: That is my partner’s job. I restore old maps.\n\nQuestion: What is the user’s work?'}], [{'role': 'system', 'content': 'Answer only from supplied evidence. Distinguish user statements from assistant drafts. Respect explicit forget requests. If evidence is insufficient, say so. Cite [E1].'}, {'role': 'user', 'content': 'Evidence [E1]:\nuser: In my screenplay, the character says: I own a vineyard in Mendoza.\n\nQuestion: Does this establish that the user owns a vineyard?'}]]
paired = [[{'role': 'system', 'content': 'Answer only from supplied evidence. Distinguish user statements from assistant drafts. Respect explicit forget requests. If evidence is insufficient, say so. Cite [E1].'}, {'role': 'user', 'content': 'Evidence [E1]:\nuser: In a radio play I wrote, the narrator says, “I tune church organs.”\n\nQuestion: Does the evidence establish that the user tunes church organs?'}], [{'role': 'system', 'content': 'Answer only from supplied evidence. Distinguish user statements from assistant drafts. Respect explicit forget requests. If evidence is insufficient, say so. Cite [E1].'}, {'role': 'user', 'content': 'Evidence [E1]:\nuser: I tune church organs.\n\nQuestion: Does the evidence establish that the user tunes church organs?'}], [{'role': 'system', 'content': 'Answer only from supplied evidence. Distinguish user statements from assistant drafts. Respect explicit forget requests. If evidence is insufficient, say so. Cite [E1].'}, {'role': 'user', 'content': 'Evidence [E1]:\nuser: My comic follows a courier. Her opening line is “I live above a puppet theatre.”\n\nQuestion: Where does the evidence say the user lives?'}], [{'role': 'system', 'content': 'Answer only from supplied evidence. Distinguish user statements from assistant drafts. Respect explicit forget requests. If evidence is insufficient, say so. Cite [E1].'}, {'role': 'user', 'content': 'Evidence [E1]:\nuser: My comic follows a courier who lives above a puppet theatre. That part comes from my life: I live above a puppet theatre too.\n\nQuestion: Where does the evidence say the user lives?'}], [{'role': 'system', 'content': 'Answer only from supplied evidence. Distinguish user statements from assistant drafts. Respect explicit forget requests. If evidence is insufficient, say so. Cite [E1].'}, {'role': 'user', 'content': 'Evidence [E1]:\nuser: Use a window seat when planning trips for me.\nuser: Please forget that seating preference.\n\nQuestion: Should the assistant use the earlier seating preference when planning the next trip?'}], [{'role': 'system', 'content': 'Answer only from supplied evidence. Distinguish user statements from assistant drafts. Respect explicit forget requests. If evidence is insufficient, say so. Cite [E1].'}, {'role': 'user', 'content': 'Evidence [E1]:\nuser: Use a window seat when planning trips for me.\nuser: Please forget the departure time I gave earlier.\n\nQuestion: Should the assistant use the earlier seating preference when planning the next trip?'}]]
def prompt(row):
    text = tok.apply_chat_template(row, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return tok.encode(text, add_special_tokens=False)

def evaluate(model, name, dev):
    model.eval()
    answers = []
    with torch.inference_mode():
        for row in dev:
            x = torch.tensor([prompt(row)], device='cuda')
            start = time.monotonic()
            y = model.generate(x, attention_mask=torch.ones_like(x), max_new_tokens=180,
                               do_sample=False, pad_token_id=tok.eos_token_id)
            answer = tok.decode(y[0, x.shape[1]:], skip_special_tokens=True)
            answers.append({'answer': answer, 'seconds': time.monotonic()-start})
    (out / (name+'.json')).write_text(json.dumps(answers, indent=2))
    print(name, json.dumps([a['answer'] for a in answers]), flush=True)

evaluate(base, 'paired_base', paired)
adapter = PeftModel.from_pretrained(base, weights.parent)
evaluate(adapter, 'paired_adapter', paired)


In [ ]:
from copy import deepcopy
instruction = (" Attribute each fact to the person who asserts it. A narrator, character, quoted speaker, "
               "or draft may speak in the first person. Treat a detail as the user's own only when the "
               "user asserts or confirms it about themselves. Apply withdrawals to the detail withdrawn.")
revised = deepcopy(original + paired)
for row in revised:
    row[0]['content'] += instruction
with adapter.disable_adapter():
    evaluate(adapter, 'revised_base', revised)
evaluate(adapter, 'revised_adapter', revised)
print('FOLLOWUP_COMPLETE', flush=True)
